# Diagnóstico Bronze — steam_games_batch

Tabla analizada: `steam_game_intelligence.bronze.steam_games_batch`. Revisa nulos, duplicados, rangos imposibles del batch actual. El esquema esperado se basa en el archivo de definición que compartiste.

> Ejecuta las celdas en orden. Estas consultas son de diagnóstico: no modifican las tablas Bronze.

## 1. Volumen y ventana de ingesta

In [0]:
%sql
SELECT
  COUNT(*) AS total_rows,
  COUNT(DISTINCT app_id) AS distinct_app_ids,
  MIN(_ingested_at) AS first_ingested_at,
  MAX(_ingested_at) AS last_ingested_at,
  COUNT_IF(app_id IS NULL) AS null_app_id_rows,
  COUNT_IF(name IS NULL OR TRIM(name) = '') AS missing_name_rows
FROM steam_game_intelligence.bronze.steam_games_batch;

## 2. Nulos por columna
La salida muestra únicamente columnas con nulos. En campos de texto también conviene distinguir NULL de cadenas vacías.

In [0]:
%sql
SELECT column_name, null_count
FROM (
  SELECT
    COUNT_IF(app_id IS NULL) AS app_id,
    COUNT_IF(name IS NULL) AS name,
    COUNT_IF(release_date IS NULL) AS release_date,
    COUNT_IF(price IS NULL) AS price,
    COUNT_IF(price_status IS NULL) AS price_status,
    COUNT_IF(estimated_owners IS NULL) AS estimated_owners,
    COUNT_IF(developers IS NULL) AS developers,
    COUNT_IF(publishers IS NULL) AS publishers,
    COUNT_IF(genres IS NULL) AS genres,
    COUNT_IF(categories IS NULL) AS categories,
    COUNT_IF(tags IS NULL) AS tags,
    COUNT_IF(positive IS NULL) AS positive,
    COUNT_IF(negative IS NULL) AS negative,
    COUNT_IF(recommendations IS NULL) AS recommendations,
    COUNT_IF(peak_ccu IS NULL) AS peak_ccu,
    COUNT_IF(metacritic_score IS NULL) AS metacritic_score,
    COUNT_IF(user_score IS NULL) AS user_score,
    COUNT_IF(average_playtime_forever IS NULL) AS average_playtime_forever,
    COUNT_IF(median_playtime_forever IS NULL) AS median_playtime_forever,
    COUNT_IF(average_playtime_2weeks IS NULL) AS average_playtime_2weeks,
    COUNT_IF(median_playtime_2weeks IS NULL) AS median_playtime_2weeks,
    COUNT_IF(short_description IS NULL) AS short_description,
    COUNT_IF(about_the_game IS NULL) AS about_the_game,
    COUNT_IF(detailed_description IS NULL) AS detailed_description,
    COUNT_IF(notes IS NULL) AS notes,
    COUNT_IF(achievements IS NULL) AS achievements,
    COUNT_IF(dlc_count IS NULL) AS dlc_count,
    COUNT_IF(packages IS NULL) AS packages,
    COUNT_IF(supported_languages IS NULL) AS supported_languages,
    COUNT_IF(full_audio_languages IS NULL) AS full_audio_languages,
    COUNT_IF(windows IS NULL) AS windows,
    COUNT_IF(mac IS NULL) AS mac,
    COUNT_IF(linux IS NULL) AS linux,
    COUNT_IF(steam_store_available IS NULL) AS steam_store_available,
    COUNT_IF(steam_spy_available IS NULL) AS steam_spy_available,
    COUNT_IF(_ingested_at IS NULL) AS _ingested_at,
    COUNT_IF(_source IS NULL) AS _source
  FROM steam_game_intelligence.bronze.steam_games_batch
) UNPIVOT (null_count FOR column_name IN (
  app_id, name, release_date, price, price_status, estimated_owners,
  developers, publishers, genres, categories, tags, positive, negative,
  recommendations, peak_ccu, metacritic_score, user_score,
  average_playtime_forever, median_playtime_forever, average_playtime_2weeks,
  median_playtime_2weeks, short_description, about_the_game, detailed_description,
  notes, achievements, dlc_count, packages, supported_languages,
  full_audio_languages, windows, mac, linux, steam_store_available,
  steam_spy_available, _ingested_at, _source
))
WHERE null_count > 0
ORDER BY null_count DESC;

## 3. Claves repetidas
Como la tabla se reemplaza con cada batch diario, que `app_id` aparezca varias veces dentro del batch puede ser un duplicado incorrecto. La segunda consulta revisa la repetición de la clave dentro de la carga actual.

## 4. Valores fuera de rango o sospechosos
Los límites son reglas de diagnóstico, no afirmaciones de que todo valor fuera de ellos sea inválido.

In [0]:
%sql
SELECT
  COUNT_IF(app_id <= 0) AS invalid_app_id,
  COUNT_IF(price < 0) AS negative_price,
  COUNT_IF(positive < 0) AS negative_positive_reviews,
  COUNT_IF(negative < 0) AS negative_negative_reviews,
  COUNT_IF(recommendations < 0) AS negative_recommendations,
  COUNT_IF(peak_ccu < 0) AS negative_peak_ccu,
  COUNT_IF(metacritic_score < 0 OR metacritic_score > 100) AS metacritic_out_of_range,
  COUNT_IF(user_score < 0 OR user_score > 100) AS user_score_out_of_range,
  COUNT_IF(achievements < 0) AS negative_achievements,
  COUNT_IF(dlc_count < 0) AS negative_dlc_count,
  COUNT_IF(release_date > CURRENT_DATE()) AS future_release_date,
  COUNT_IF(_ingested_at IS NULL) AS missing_ingestion_timestamp
FROM steam_game_intelligence.bronze.steam_games_batch;

In [0]:
%sql
SELECT app_id, name, release_date, price, positive, negative,
       recommendations, peak_ccu, metacritic_score, user_score,
       achievements, dlc_count, _ingested_at
FROM steam_game_intelligence.bronze.steam_games_batch
WHERE app_id <= 0
   OR price < 0
   OR positive < 0 OR negative < 0
   OR recommendations < 0 OR peak_ccu < 0
   OR metacritic_score < 0 OR metacritic_score > 100
   OR user_score < 0 OR user_score > 100
   OR achievements < 0 OR dlc_count < 0
   OR release_date > CURRENT_DATE()
   OR name IS NULL OR TRIM(name) = ''
LIMIT 200;

## 5. Cadenas vacías y normalización básica

In [0]:
%sql
SELECT
  COUNT_IF(name IS NOT NULL AND TRIM(name) = '') AS blank_name,
  COUNT_IF(developers IS NOT NULL AND TRIM(developers) = '') AS blank_developers,
  COUNT_IF(publishers IS NOT NULL AND TRIM(publishers) = '') AS blank_publishers,
  COUNT_IF(genres IS NOT NULL AND TRIM(genres) = '') AS blank_genres,
  COUNT_IF(categories IS NOT NULL AND TRIM(categories) = '') AS blank_categories,
  COUNT_IF(_source IS NOT NULL AND TRIM(_source) = '') AS blank_source
FROM steam_game_intelligence.bronze.steam_games_batch;

## 6. Variaciones entre batchs
Detecta app_id cuyo nombre o precio cambió entre ingestas. Es información útil para decidir cómo construir Silver; no se debe borrar el historial sin definir primero el objetivo.

In [0]:
%sql
SELECT app_id,
       COUNT(DISTINCT name) AS distinct_names,
       COUNT(DISTINCT price) AS distinct_prices,
       COUNT(DISTINCT release_date) AS distinct_release_dates,
       COUNT(DISTINCT genres) AS distinct_genre_values,
       COUNT(*) AS batchs
FROM steam_game_intelligence.bronze.steam_games_batch
WHERE app_id IS NOT NULL
GROUP BY app_id
HAVING COUNT(DISTINCT name) > 1
    OR COUNT(DISTINCT price) > 1
    OR COUNT(DISTINCT release_date) > 1
    OR COUNT(DISTINCT genres) > 1
ORDER BY batchs DESC
LIMIT 100;